# 108장 · 평가 문항 중복 제거와 오답 분석

문항 ID가 달라도 질문 내용이 같으면 평가 데이터가 겹칠 수 있습니다. 표기를 정리한 뒤 개발 문항과 확인용 문항을 대조합니다.

개발 질문과 확인용 후보의 중복을 검사합니다. 문항 ID나 공백만 다른 질문을 제외하고, 오답의 원문·기대값·실제값을 나란히 기록합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 팀별 변형을 시도한 뒤 해당 셀 아래에서 예시를 펼쳐 비교할 수 있습니다.


## 정답 원문과 실제 화면을 따로 대조하기

90장에서 실행 전에 고정한 문서 ID와 정답 문장을 꺼냅니다. 사용자 과업 결과는 그대로 두고 실제 화면의 출처와 답변을 채점합니다. 다른 장비의 문장이 표시되면 어느 항목이 실패할까요?

이 프로젝트의 노트북은 한 폴더에서 열어 주세요. 첫 셀은 `team-project/final/chapter107.json`을 먼저 찾습니다. 팀 파일이 없으면 [이 장의 시작 예시](chapter108-start.json)를 노트북 옆에 내려받으세요. 출력 첫 줄에서 어느 파일을 읽었는지 확인할 수 있습니다.

In [ ]:
from pathlib import Path
import json
import re

here = Path.cwd()
project_root = here.parent if re.fullmatch(r"chapter[0-9]+", here.name) else here
team_file = project_root / "team-project/final/chapter107.json"
sample_file = here / "chapter108-start.json"
input_file = team_file if team_file.is_file() else sample_file
if not input_file.is_file():
    raise FileNotFoundError("이 장의 시작 예시를 내려받아 노트북 옆에 두세요.")
incoming = json.loads(input_file.read_text(encoding="utf-8"))
print("팀 기록" if input_file == team_file else "교재 예시", input_file.name)
print(incoming['quality_reference'])

교재 예시 chapter108-start.json
{'case': 'Q01', 'question': '장비 A의 점검 전 확인 사항은?', 'id': 'RULE-A', 'fact': '점검 전 전원 표시등을 확인합니다.'}


두 번째 줄은 읽은 파일의 `quality_reference` 값입니다.

## 질문 문자열 정규화와 중복 판별

준비 자료의 PUBLIC-DUP-01은 Q01 문장의 양 끝에 공백을 붙였습니다. 이름이 새로워도 이미 본 문장입니다. PUBLIC-PARAPHRASE-01의 “장비 A를 살펴보기 전에 볼 것은 무엇인가요?”처럼 글자만 달라진 문항은 사람도 의미 중복을 살펴야 합니다.


### 셀 01 · 질문 정규화와 중복 판별

개발 질문과 공개 후보 하나를 문자열 수준에서 대조합니다.

‘장비 A 점검 전에 볼 것은?’과 ‘장비 A는 점검 전 무엇을 확인하나요?’는 표기가 달라도 의도와 기대 근거 RULE-A가 같습니다. 확인용 세트에 넣기 전에 같은 능력을 다시 묻는 문항인지 검토하고 선택 이유를 남기세요.


In [1]:
development = '장비 A의 점검 전 확인 사항은?'
candidate = '  장비 A의 점검 전 확인 사항은?  '
print(candidate.strip() == development.strip())

True


True입니다. 이 후보는 독립 점수의 분모에서 제외할 수 있습니다. 문자열이 다르다는 결과만으로 의미가 새로운 질문이라고 판정할 수는 없습니다.

질문 정규화 뒤 같은 문자열이 된 두 문항을 개발용과 확인용에 나눠 넣으면 사실상 같은 입력을 두 번 평가하게 됩니다. 확인 세트를 구성할 때는 표기 차이를 제거한 뒤 중복을 검사하고, 기대 근거가 같은 이유도 원문으로 대조합니다.


## 기대 근거와 실제 응답 비교

실패 분석 연습용 가상 행　질문 “B 장비의 점검 전 기준은?” / 기대 고정 장치, RULE-B 2.2절 / 가정한 검색 결과 RULE-B / 가정한 화면 답 전원 표시등 / 판단 자료는 맞게 골랐고 출력 내용이 원문과 어긋남. 실제 실행에서 중간 결과가 없으면 ‘검색 오류’라고 단정하지 말고 관찰할 단계를 먼저 적습니다.


### 팀 참고 예시 · 기대값 불일치 기록


In [2]:
answer = "점검 전 전원 표시등을 확인합니다."
print("고정 장치" in answer)
print("조회한 원문이 답 생성 단계에도 그대로 전달됐는지 확인한다.")

False
조회한 원문이 답 생성 단계에도 그대로 전달됐는지 확인한다.


False는 답에 기대 표현이 빠졌다는 뜻입니다. 어느 함수에서 바뀌었는지 찾으려면 검색 뒤에 실제로 넘긴 자료와 화면에 도착한 답을 읽어야 합니다.


#### 정규화로 같은 질문을 모을 때

<mark>글자 모양이 비슷한 두 질문도 기대 근거가 다르면 별개 문항일 수 있습니다.</mark> 문자열 비교 뒤에 장비명과 근거 절을 대조하세요. 중복 제거로 어려운 문항만 빠지면 모델을 바꾸지 않아도 점수가 달라집니다.

선수 개념 · 40장 모델·판본을 구분하는 평가 질문 만들기

#### 검색에서 RULE-B를 찾았는데 화면 답은 왜 틀렸을까?

```mermaid
flowchart TD
nexpect["기대 기준"]
  nfound["검색 결과"]
  nanswer["화면 답"]
  ncompare["필수 사실 대조"]
  nexpect --> nfound
  nfound --> nanswer
  nanswer --> ncompare
```

가상 실패 행의 중간 결과를 따라가며 검색 단계와 답변 단계 중 어디서 기대값이 달라지는지 찾습니다.

## 검색 성공과 답변 성공을 한 문항에서 따로 채점하기

B 장비 질문이 A의 전원 표시등 안내를 내놓았습니다. 검색 결과까지 남아 있다면 규정을 못 찾은 것인지, 올바른 규정을 받고 답을 잘못 만든 것인지 판단할 수 있습니다.

### 셀 04 · 같은 Q04의 중간 결과를 정답에 맞춰 보기

이 장의 answer는 A의 문장입니다. 합성 오류를 분석하기 위해 검색은 RULE-B를 찾았다고 두고, 생성 답의 필수 사실을 따로 확인합니다.

In [3]:
expectation = {"doc": "RULE-B", "fact": "고정 장치"}
retrieved_ids = ["RULE-B"]
evidence_hit = expectation["doc"] in retrieved_ids
answer_hit = expectation["fact"] in answer
print("기대 문서 포함", evidence_hit)
print("기대 사실 포함", answer_hit)

기대 문서 포함 True
기대 사실 포함 False


첫 값만 True입니다. 검색을 다시 튜닝하기 전에 검색 결과를 프롬프트에 넣은 부분과 생성한 문장을 대조할 차례입니다. 단어 포함은 이 사례의 단순 검산이므로, 부정 표현이 있으면 사람이 의미도 확인합니다.

### 확인용 문항 관리 예시

<table><thead><tr><th scope="col">받은 후보</th><th scope="col">검토</th><th scope="col">처리</th></tr></thead><tbody><tr><td>Q01 앞뒤에 공백을 붙인 문장</td><td>정규화 후 개발 문항과 동일</td><td>회귀 묶음으로 이동</td></tr><tr><td>장비 A를 살펴보기 전에 볼 것은 무엇인가요?</td><td>표현이 달라도 같은 장비·사실을 묻는 공개 사례</td><td>독립 확인용으로 세지 않음</td></tr><tr><td>다른 담당자가 후보 동결 후 전달한 질문</td><td>작성자·판본·기대 근거 확인 필요</td><td>개발자가 보기 전 분리 보관</td></tr></tbody></table>

독립 질문이 실제로 없는 상태라면 확인용 문항 수를 0으로 기록합니다. 공개 예제 두 건을 비공개 평가로 바꾸어 부르지 않습니다.

<mark>최종 답만 남았으면 실패한 단계를 확정할 수 없습니다.</mark> 틀린 문장을 받더라도 검색 후보에 RULE-B가 있었는지에 따라 처음 살펴볼 코드가 달라집니다.

109장에는 평가 기대값과 함께 결과를 볼 수 있는 사용자 범위를 넘깁니다.

## 팀 사례에 적용하기 · 정답 원문과 실제 화면을 따로 대조하기

위 예제에 첫 셀에서 읽어 둔 팀 기록을 넣어 봅니다.

In [ ]:
truth = incoming["quality_reference"]
def grade(view, expected):
    return {"doc": expected["id"] in view["sources"],
            "fact": expected["fact"] == view["answer"]}
matches_question = incoming["reply"]["question"] == truth["question"]
score = grade(incoming["view"], truth) if matches_question else {"doc": None, "fact": None}
wrong_text = {**incoming["view"], "answer": "원문에 없는 점검 시간은 오후 세 시입니다."}
new = {**incoming, "answer_review": {**score, "case": truth["case"], "split": "공개 개발 예시"}}
print("실제 화면", score)
print("문장만 바꾼 대조", grade(wrong_text, truth))

실제 화면 {'doc': True, 'fact': True}
문장만 바꾼 대조 {'doc': True, 'fact': False}


ID는 맞지만 답의 문장만 바꾸면 문서 점수는 참, 사실 점수는 거짓이 됩니다. 이 예시는 정답 원문을 그대로 표시하는 서비스여서 문자열 일치를 씁니다. 팀 챗봇이 바꿔 쓴 답은 허용 표현이나 별도 판정이 필요합니다.

In [ ]:
output_file = project_root / "team-project/final/chapter108.json"
output_file.parent.mkdir(parents=True, exist_ok=True)
with output_file.open("a+", encoding="utf-8") as handle:
    handle.seek(0)
    previous_text = handle.read()
    if previous_text and json.loads(previous_text) != new:
        raise FileExistsError("같은 장의 다른 결과가 있습니다. 기존 JSON을 다른 이름으로 보관한 뒤 다시 실행하세요.")
    if not previous_text:
        json.dump(new, handle, ensure_ascii=False, indent=2)
        handle.write("\n")
print(output_file.relative_to(project_root).as_posix())

team-project/final/chapter108.json


109장 첫 셀이 이 파일을 읽습니다. 입력을 바꿔 다시 실행할 때는 기존 결과 파일의 이름을 바꿔 보관하세요.

## 돌아보기

확인용 자료의 보관 담당자와 열어 본 시점을 기록하세요. 이 장에 보이는 질문은 모두 공개 연습 사례입니다. 실제 비공개 확인용 질문·점수는 아직 없습니다.
